# 01. Exploratory Data Analysis (EDA) - MediSure Health Insurance

This notebook explores the **MediSure Health Insurance** warehouse dataset stored in DuckDB.
We conduct demographic profiling, financial analysis of claims, epidemiological patterns, and statistical hypothesis testing.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.utils.io import get_duckdb_connection

con = get_duckdb_connection(read_only=True)
print("Connected to MediSure DuckDB warehouse.")


## 1. Portfolio Demographic & Policy Overview

In [ ]:
members_df = con.execute("SELECT * FROM dim_member").df()
policies_df = con.execute("SELECT * FROM dim_policy").df()

print(f"Total Members: {len(members_df):,}")
print(f"Total Policies: {len(policies_df):,}")
display(members_df.head())


## 2. Incurred Claims Distribution & Log Transformation

In [ ]:
claims_df = con.execute("SELECT * FROM fact_claims").df()
paid_claims = claims_df[claims_df['paid_amount'] > 0]['paid_amount']

print(f"Total Claims: {len(claims_df):,}")
print(f"Mean Paid Amount: INR {paid_claims.mean():,.2f}")
print(f"Median Paid Amount: INR {paid_claims.median():,.2f}")
print(f"P95 Paid Amount: INR {paid_claims.quantile(0.95):,.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(paid_claims, bins=40, kde=True, ax=axes[0], color='navy')
axes[0].set_title("Paid Claim Amount Distribution (INR)")
sns.histplot(np.log1p(paid_claims), bins=40, kde=True, ax=axes[1], color='teal')
axes[1].set_title("Log-Transformed Claim Amount log1p(₹)")
plt.show()


## 3. Statistical Hypothesis Testing
### Mann-Whitney U: Cashless vs Reimbursement TAT

In [ ]:
cashless_tat = claims_df[claims_df['claim_type'] == 'Cashless']['settlement_tat_days'].dropna()
reimb_tat = claims_df[claims_df['claim_type'] == 'Reimbursement']['settlement_tat_days'].dropna()

u_stat, p_val = stats.mannwhitneyu(cashless_tat, reimb_tat)
print(f"Mann-Whitney U statistic: {u_stat:,.1f}, p-value: {p_val:.4e}")
print(f"Cashless Mean TAT: {cashless_tat.mean():.2f} days vs Reimbursement Mean TAT: {reimb_tat.mean():.2f} days")


### Kruskal-Wallis: Severity Across Hospital Tiers

In [ ]:
tier_df = con.execute("""
    SELECT c.paid_amount, p.hospital_tier
    FROM fact_claims c
    JOIN dim_provider p ON c.provider_id = p.provider_id
    WHERE c.paid_amount > 0
""").df()

h_stat, kw_pval = stats.kruskal(
    tier_df[tier_df['hospital_tier'] == 'Tier1']['paid_amount'],
    tier_df[tier_df['hospital_tier'] == 'Tier2']['paid_amount'],
    tier_df[tier_df['hospital_tier'] == 'Tier3']['paid_amount']
)
print(f"Kruskal-Wallis H-statistic: {h_stat:,.2f}, p-value: {kw_pval:.4e}")


In [ ]:
con.close()